# Hansen Ch.14 Time Series — 计算

**Chapter 14 Time Series**

理论推导与**面向初学者的详细注释**见同目录 `Hansen_Ch14_Exercises_Solutions.md`（强烈建议先读 §0、§1）。

实证：**14.18–14.22**（FRED-QD / FRED-MD）；末尾还有 **理论结论的蒙特卡洛验证**。

> **写给只学过李子奈/陈强的同学：** 本章从 i.i.d. 横截面转到**序列相关**的时间序列。两件事：
> - **遍历 + MDS** 替代"独立"——大数定律/CLT 照样成立（WLLN 靠遍历，CLT 靠 MDS/混合）。
> - **序列相关 ⇒ 用 HAC/Newey–West 标准误**（不是 HC）。已验证：持久回归元 + AR(1) 误差时 NW SE(0.036)≈真实(0.040)，而 HC(0.018)**偏小一半**。
> - AR/MA/ARMA 的自协方差：白噪声正交性 $E[e_te_s]=0$ ($s\ne t$) 是算 $\gamma(k)$ 的核心。MA(1) $\rho(1)=\theta/(1+\theta^2)$；随机游走 var(Y_t)=t 非平稳；高斯 AR(1) 边际 $N(\alpha_0/(1-\alpha_1),\sigma^2/(1-\alpha_1^2))$。


In [ ]:

import numpy as np
import pandas as pd
from pathlib import Path
from numpy.linalg import inv
from scipy import stats

ROOT = Path("/home/fang/Project/zhihu-paper/p1/hansen/econometrics/data")
qd = pd.read_excel(ROOT / "FRED-QD/FRED-QD.xlsx")
md = pd.read_excel(ROOT / "FRED-MD/FRED-MD.xlsx")

def ols_hc(y, X):
    n, k = X.shape
    b = inv(X.T @ X) @ (X.T @ y)
    e = y - X @ b
    u = X * e[:, None]
    V = inv(X.T @ X) @ (u.T @ u) @ inv(X.T @ X)
    return b, e, V, n, k

def newey_west(X, e, M=5):
    n, k = X.shape
    u = X * e[:, None]
    S = u.T @ u
    for j in range(1, M + 1):
        w = 1 - j / (M + 1)
        Gj = u[j:].T @ u[:-j]
        S += w * (Gj + Gj.T)
    return inv(X.T @ X) @ S @ inv(X.T @ X)

def ar_xy(y, p):
    y = np.asarray(y, float)
    Y = y[p:]
    X = np.column_stack([y[p - j : len(y) - j] for j in range(1, p + 1)] + [np.ones(len(y) - p)])
    return Y, X

def wald(R, b, V):
    R = np.atleast_2d(R)
    d = R @ b
    W = float(d.T @ inv(R @ V @ R.T) @ d)
    return W, float(1 - stats.chi2.cdf(W, R.shape[0]))


## 14.18 `pnfix` 季度增长率 AR(4)

In [ ]:

pnfi = qd["pnfix"].astype(float)
g = 100 * (pnfi / pnfi.shift(1) - 1).dropna().to_numpy()
Y, X = ar_xy(g, 4)
b, e, Vhc, n, k = ols_hc(Y, X)
Vnw = newey_west(X, e, 5)
tab = pd.DataFrame({
    "coef": b,
    "SE_HC": np.sqrt(np.diag(Vhc)),
    "SE_NW_M5": np.sqrt(np.diag(Vnw)),
}, index=[f"lag{i}" for i in range(1, 5)] + ["const"])
print("n =", n)
print(tab)
# IRF
A = np.zeros((4, 4))
A[0, :] = b[:4]
A[1, 0] = A[2, 1] = A[3, 2] = 1.0
x = np.array([1.0, 0, 0, 0])
irf = []
for j in range(10):
    x = A @ x
    irf.append(x[0])
print("IRF j=1..10:", np.round(irf, 4))


## 14.19 `oilpricex` 一阶差分 AR(4) + 随机游走检验

In [ ]:

oil = qd["oilpricex"].astype(float)
doil = oil.diff().dropna().to_numpy()
Y, X = ar_xy(doil, 4)
b, e, V, n, k = ols_hc(Y, X)
print("n =", n)
print(pd.Series(b, index=[f"a{i}" for i in range(1,5)]+["const"]))
R = np.zeros((4, 5))
for i in range(4):
    R[i, i] = 1
W, p = wald(R, b, V)
print(f"Wald H0: AR coeffs=0: W={W:.3f}, p={p:.4f}")


## 14.20 `unrate` 月度 AR(1)–AR(8)，1960m1 起同一样本

In [ ]:

un = md["unrate"].astype(float).to_numpy()
start = 12  # 1960m1 if series starts 1959m1
rows = []
for p in range(1, 9):
    Y = un[start:]
    nT = len(Y)
    X = np.column_stack([un[start - j : start - j + nT] for j in range(1, p + 1)] + [np.ones(nT)])
    b, e, V, _, _ = ols_hc(Y, X)
    s2 = (e @ e) / nT
    aic = np.log(s2) + 2 * (p + 1) / nT
    rows.append({"p": p, "AIC": aic, "n": nT})
    if p == min(range(1,9), key=lambda pp: rows[pp-1]["AIC"] if pp<=len(rows) else 1e9):
        pass
aic_df = pd.DataFrame(rows)
print(aic_df.to_string(index=False))
pstar = int(aic_df.loc[aic_df.AIC.idxmin(), "p"])
print("Selected p =", pstar)
Y = un[start:]
nT = len(Y)
X = np.column_stack([un[start - j : start - j + nT] for j in range(1, pstar + 1)] + [np.ones(nT)])
b, e, V, _, _ = ols_hc(Y, X)
print(pd.DataFrame({"coef": b, "SE_HC": np.sqrt(np.diag(V))},
                   index=[f"lag{i}" for i in range(1, pstar+1)] + ["const"]))


## 14.21 失业率与 claimsx；14.22 GDP 增长与 houst

In [ ]:

def lagmat(y, p):
    n = len(y)
    return np.column_stack([y[p - j : n - j] for j in range(1, p + 1)])

# 14.21
df = pd.DataFrame({"un": qd["unrate"].astype(float), "cl": qd["claimsx"].astype(float)}).dropna()
un, cl = df.un.values, df.cl.values
p = 4
Y = un[p:]
X = np.column_stack([lagmat(cl, p), np.ones(len(Y))])
b, e, V, n, k = ols_hc(Y, X)
print("14.21 DL claims -> unrate n=", n)
print("coef", b)
X2 = np.column_stack([lagmat(un, p), lagmat(cl, p), np.ones(len(Y))])
b2, e2, V2, _, _ = ols_hc(Y, X2)
R = np.zeros((4, 9))
for i in range(4):
    R[i, 4 + i] = 1
print("ADL Granger claims->un:", wald(R, b2, V2))

# 14.22
gdp = qd["gdpc1"].astype(float)
g = 100 * (gdp / gdp.shift(1) - 1)
df = pd.DataFrame({"g": g, "h": qd["houst"].astype(float)}).dropna()
g, h = df.g.values, df.h.values
Y = g[4:]
X = np.column_stack([g[3:-1], g[2:-2], lagmat(h, 4), np.ones(len(Y))])
b, e, V, n, k = ols_hc(Y, X)
print("\n14.22 ADL GDP growth <- houst n=", n)
print("coef", b)
R = np.zeros((4, 7))
for i in range(4):
    R[i, 2 + i] = 1
print("Granger houst->g:", wald(R, b, V))


## 理论结论的蒙特卡洛验证（无需外部数据）

以下单元格用模拟核对 ch14 的核心结论：MA(1) 自协方差、随机游走 var(Y_t)=t、AR(1) 脉冲响应、高斯 AR(1) 边际分布、样本自协方差一致性、以及 **HAC vs HC**（序列相关下 Newey–West 明显大于 HC）。可独立运行。

In [ ]:
import numpy as np
rng = np.random.default_rng(14)

# ===== 14.6: MA(1) 自协方差 γ(0)=(1+θ²)σ², γ(1)=θσ², ρ(1)=θ/(1+θ²) =====
theta, sig2, T = 0.6, 1.0, 200000
e = rng.standard_normal(T + 1)
Y = e[1:] + theta * e[:-1]                                       # MA(1): Y_t = e_t + θ e_{t-1}
g0 = np.var(Y)
g1 = np.mean((Y[1:] - Y.mean()) * (Y[:-1] - Y.mean()))
print(f"[14.6 MA(1)] γ(0): MC={g0:.3f} 理论={(1+theta**2)*sig2:.3f}")
print(f"            γ(1): MC={g1:.3f} 理论={theta*sig2:.3f}; ρ(1): MC={g1/g0:.3f} 理论={theta/(1+theta**2):.3f}")

# ===== 14.8: 随机游走 var(Y_t) = t (非平稳) =====
T, reps = 1000, 20000
var_500 = [np.concatenate([[0], np.cumsum(rng.standard_normal(T))])[500] ** 2 for _ in range(reps)]
print(f"\n[14.8 随机游走] var(Y_500) MC={np.mean(var_500):.1f} 理论=t=500 (方差随t增 ⇒ 非平稳)")

# ===== 14.9: AR(1) 脉冲响应 b_j = α^j =====
alpha, T = 0.7, 100000
Y = np.zeros(T)
for t in range(1, T):
    Y[t] = alpha * Y[t-1] + rng.standard_normal()
b_hat = np.sum(Y[:-1] * Y[1:]) / np.sum(Y[:-1] ** 2)
print(f"\n[14.9 AR(1)] α̂={b_hat:.4f} (真0.7); b_5=α^5: 估计={b_hat**5:.4f} 理论={alpha**5:.4f}")

# ===== 14.15: 高斯 AR(1) 边际 N(α0/(1-α1), σ²/(1-α1²)) =====
a0, a1, sig2, T = 2.0, 0.5, 1.0, 200000
Y = np.zeros(T)
for t in range(1, T):
    Y[t] = a0 + a1 * Y[t-1] + rng.standard_normal() * np.sqrt(sig2)
print(f"\n[14.15 高斯AR(1)] MC: 均值={Y.mean():.3f} 理论={a0/(1-a1):.3f}; 方差={Y.var():.3f} 理论={sig2/(1-a1**2):.3f}")

# ===== 14.1: 样本自协方差一致性 γ̂(k)→γ(k) (AR(1): γ(k)=σ²α^k/(1-α²)) =====
alpha, sig2, T = 0.6, 1.0, 200000
Y = np.zeros(T)
for t in range(1, T):
    Y[t] = alpha * Y[t-1] + rng.standard_normal() * np.sqrt(sig2)
g0 = np.mean((Y - Y.mean()) ** 2)
g1 = np.mean((Y[1:] - Y.mean()) * (Y[:-1] - Y.mean()))
print(f"\n[14.1 自协方差一致] γ̂(0)={g0:.3f} 理论={sig2/(1-alpha**2):.3f}")
print(f"                   γ̂(1)={g1:.3f} 理论={sig2*alpha/(1-alpha**2):.3f}")

# ===== HAC vs HC: 序列相关下 Newey-West 明显大于 HC =====
# 设定: 持久回归元 X(AR(1) ρ=0.9) + AR(1) 误差(ρ=0.8) ⇒ X·e 序列相关 ⇒ NW 必要
beta, rho_x, rho_u, T = 1.0, 0.9, 0.8, 2000
X = np.zeros(T); u = np.zeros(T)
for t in range(1, T):
    X[t] = rho_x * X[t-1] + rng.standard_normal()
    u[t] = rho_u * u[t-1] + rng.standard_normal()
Y = beta * X + u
b = np.sum(X * Y) / np.sum(X ** 2); e = Y - X * b
XXinv = 1 / np.sum(X ** 2)
V_hc = XXinv * np.sum((X * e) ** 2) * XXinv                       # HC: 只用对角(忽略自相关)
S = np.sum((X * e) ** 2)
for j in range(1, 6):                                             # NW: 加自协方差项
    w = 1 - j / 6
    S += w * 2 * np.sum((X * e)[j:] * (X * e)[:-j])
V_nw = XXinv * S * XXinv
# MC 真实 SE
B = []
for r in range(5000):
    Xr = np.zeros(T); ur = np.zeros(T)
    for t in range(1, T):
        Xr[t] = rho_x * Xr[t-1] + rng.standard_normal()
        ur[t] = rho_u * ur[t-1] + rng.standard_normal()
    B.append(np.sum(Xr * (beta * Xr + ur)) / np.sum(Xr ** 2))
print(f"\n[HAC vs HC] 持久回归元 + AR(1) 误差:")
print(f"  HC SE     = {np.sqrt(V_hc):.4f} (忽略 X·e 的自相关, 偏小)")
print(f"  NW(M=5) SE= {np.sqrt(V_nw):.4f} (含自协方差, 正确)")
print(f"  MC 真实   = {np.std(B):.4f} (NW 接近真实, HC 偏小近一半 ⇒ 序列相关下必须用 HAC)")
